In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt  # ← 追加（グラフ描画用）

# =============================================
# 既存の全体スコア計算関数（変更なし）
# =============================================
def calculate_metrics(file_path: str) -> dict:
    df = pd.read_csv(file_path)
    result = {}

    # ===== 頭のブレ（head_movement）=====
    try:
        head = df[df["landmark_index"] == 0].copy()
        head["diff"] = np.sqrt(
            head["x"].diff()**2 + head["y"].diff()**2 + head["z"].diff()**2
        )
        result["head_movement"] = head["diff"].mean(skipna=True)
    except Exception as e:
        print(f"⚠️ head_movement error ({file_path}): {e}")
        result["head_movement"] = np.nan

    # ===== 肩の傾き（shoulder_tilt）=====
    try:
        left_shoulder = df[df["landmark_index"] == 11]
        right_shoulder = df[df["landmark_index"] == 12]
        result["shoulder_tilt"] = abs(left_shoulder["y"].values - right_shoulder["y"].values).mean()
    except Exception:
        result["shoulder_tilt"] = np.nan

    # ===== 体幹の傾き（torso_tilt）=====
    try:
        left_hip = df[df["landmark_index"] == 23]
        right_hip = df[df["landmark_index"] == 24]
        result["torso_tilt"] = abs(left_hip["y"].values - right_hip["y"].values).mean()
    except Exception:
        result["torso_tilt"] = np.nan

    # ===== 足上げ高さ（leg_lift）=====
    try:
        hip = df[df["landmark_index"] == 24]
        ankle = df[df["landmark_index"] == 28]
        result["leg_lift"] = (ankle["y"] - hip["y"]).min()
    except Exception:
        result["leg_lift"] = np.nan

    # ===== 接地足の横ブレ（foot_sway）=====
    try:
        foot = df[df["landmark_index"] == 28]
        result["foot_sway"] = foot["x"].std(skipna=True)
    except Exception:
        result["foot_sway"] = np.nan

    # ===== 腕の垂れ下がり（arm_sag）=====
    try:
        shoulder = df[df["landmark_index"] == 12]
        wrist = df[df["landmark_index"] == 16]
        result["arm_sag"] = (wrist["y"] - shoulder["y"]).mean()
    except Exception:
        result["arm_sag"] = np.nan

    return result


# =============================================
# 🆕 追加：フレームごとのスコアを計算する関数
# =============================================
def calculate_metrics_by_frame(file_path: str) -> pd.DataFrame:
    """
    各フレームごとに6項目のスコアを算出
    """
    df = pd.read_csv(file_path)
    frames = sorted(df["frame"].unique())
    results = []

    for frame in frames:
        frame_df = df[df["frame"] == frame]

        try:
            # 頭のブレ（frame単位）
            head = frame_df[frame_df["landmark_index"] == 0]
            head_movement = np.sqrt(
                (head["x"].diff()**2 + head["y"].diff()**2 + head["z"].diff()**2)
            ).mean(skipna=True)

            # 肩の傾き
            left_shoulder = frame_df[frame_df["landmark_index"] == 11]
            right_shoulder = frame_df[frame_df["landmark_index"] == 12]
            shoulder_tilt = abs(left_shoulder["y"].values - right_shoulder["y"].values).mean()

            # 体幹の傾き
            left_hip = frame_df[frame_df["landmark_index"] == 23]
            right_hip = frame_df[frame_df["landmark_index"] == 24]
            torso_tilt = abs(left_hip["y"].values - right_hip["y"].values).mean()

            # 足上げ高さ
            hip = frame_df[frame_df["landmark_index"] == 24]
            ankle = frame_df[frame_df["landmark_index"] == 28]
            leg_lift = (ankle["y"].values - hip["y"].values).mean()

            # 足の横ブレ
            foot = frame_df[frame_df["landmark_index"] == 28]
            foot_sway = foot["x"].std(skipna=True)

            # 腕の垂れ下がり
            shoulder = frame_df[frame_df["landmark_index"] == 12]
            wrist = frame_df[frame_df["landmark_index"] == 16]
            arm_sag = (wrist["y"].values - shoulder["y"]).mean()

        except Exception:
            head_movement = shoulder_tilt = torso_tilt = leg_lift = foot_sway = arm_sag = np.nan

        results.append({
            "frame": frame,
            "head_movement": head_movement,
            "shoulder_tilt": shoulder_tilt,
            "torso_tilt": torso_tilt,
            "leg_lift": leg_lift,
            "foot_sway": foot_sway,
            "arm_sag": arm_sag
        })

    return pd.DataFrame(results)


# =============================================
# 🆕 追加：フレームごとの結果を可視化する関数
# =============================================
def plot_frame_metrics(df_frame: pd.DataFrame, title="Frame-wise Motion Dynamics"):
    """
    各項目のフレーム変化を折れ線グラフで可視化
    """
    plt.figure(figsize=(10, 6))
    for col in ["head_movement", "shoulder_tilt", "torso_tilt", "leg_lift", "foot_sway", "arm_sag"]:
        plt.plot(df_frame["frame"], df_frame[col], label=col)
    plt.xlabel("Frame", fontsize=12)
    plt.ylabel("Value", fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()